LDA code heavily adapted from the [following tutorial](https://www.geeksforgeeks.org/machine-learning/latent-dirichlet-allocation-and-topic-modelling/)

Install dependencies and load data

In [1]:
!pip install --upgrade gensim pyLDAvis spacy pandas scikit-learn

     ---------------------------------------- 15.4/15.4 MB 9.0 MB/s eta 0:00:00
     ---------------------------------------- 9.9/9.9 MB 33.3 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.3.3
    Uninstalling pandas-2.3.3:
      Successfully uninstalled pandas-2.3.3


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\willi\\CPSC449\\canada-oil-greenwash-scraping\\greenwash-scraping\\Lib\\site-packages\\~~ndas\\_libs\\algos.cp311-win_amd64.pyd'
Check the permissions.


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import spacy.cli
spacy.cli.download("en_core_web_md")

import pandas as pd
import warnings
import string
import spacy
import nltk
import gensim
import matplotlib.pyplot as plt
from gensim import corpora
from gensim.models import CoherenceModel
from gensim.models import Phrases
from gensim.models.phrases import Phraser
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis
from nltk.corpus import stopwords
import en_core_web_md
nltk.download('wordnet')
nltk.download('stopwords')
pyLDAvis.enable_notebook()
warnings.filterwarnings("ignore")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\willi\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\willi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
green_claims = pd.read_csv('../../output/analyzed/vagueness_analyzed.csv')

# Split by time period
groups_wayback = {
    "Wayback":     green_claims[green_claims['isWayback'] == True].copy(),
    "Non-Wayback": green_claims[green_claims['isWayback'] == False].copy(),
}
print("=== isWayback split ===")
for label, df in groups_wayback.items():
    print(f"  {label}: {len(df):,} claims")

# Split by organization
orgs = sorted(green_claims["Organization"].dropna().unique())
groups_org = {org: green_claims[green_claims['Organization'] == org].copy() for org in orgs}
print("\n=== Organization split ===")
for label, df in groups_org.items():
    print(f"  {label}: {len(df):,} claims")

# Split by organization and time period
groups_combined = {}
print("\n=== Organization + isWayback split ===")
for org in orgs:
    for wb_val, wb_label in [(True, "Wayback"), (False, "Non-Wayback")]:
        label = f"{org} | {wb_label}"
        sub = green_claims[
            (green_claims['Organization'] == org) &
            (green_claims['isWayback'] == wb_val)
        ].copy()
        if len(sub) > 0:
            groups_combined[label] = sub
            print(f"  {label}: {len(sub):,} claims")
        else:
            print(f"  Skipping empty group: {label}")

=== isWayback split ===
  Wayback: 4,141 claims
  Non-Wayback: 1,680 claims

=== Organization split ===
  Canadian Natural Resources: 575 claims
  Enbridge: 1,660 claims
  Imperial Oil: 469 claims
  Pembina Pipeline: 1,143 claims
  Shell Canada: 573 claims
  Suncor Energy: 1,401 claims

=== Organization + isWayback split ===
  Canadian Natural Resources | Wayback: 487 claims
  Canadian Natural Resources | Non-Wayback: 88 claims
  Enbridge | Wayback: 833 claims
  Enbridge | Non-Wayback: 827 claims
  Imperial Oil | Wayback: 411 claims
  Imperial Oil | Non-Wayback: 58 claims
  Pembina Pipeline | Wayback: 544 claims
  Pembina Pipeline | Non-Wayback: 599 claims
  Shell Canada | Wayback: 536 claims
  Shell Canada | Non-Wayback: 37 claims
  Suncor Energy | Wayback: 1,330 claims
  Suncor Energy | Non-Wayback: 71 claims


Preprocessing

In [16]:
# Remove common words and company names
stop_words = set(stopwords.words("english")) | {
    "suncor",
    "shell",
    "pembina",
    "enbridge",
    "imperial",
}


def remove_stopwords(text):
    return " ".join([w for w in text.split() if w.lower() not in stop_words])

In [17]:
def clean_text(text):
    delete_dict = {sp_char: '' for sp_char in string.punctuation}
    delete_dict[' '] = ' '
    table = str.maketrans(delete_dict)
    text1 = text.translate(table)
    textArr = text1.split()
    text2 = ' '.join([w for w in textArr if not w.isdigit() and len(w) > 3])
    return text2.lower()

In [18]:
nlp = en_core_web_md.load(disable=['parser', 'ner'])

def lemmatization(texts, allowed_postags=['NOUN', 'ADJ']):
    output = []
    for sent in texts:
        doc = nlp(sent)
        output.append(
            [token.lemma_.lower() for token in doc if token.pos_ in allowed_postags and token.lemma_.lower() not in stop_words])
    return output

In [19]:
def preprocess(df):
    df = df.copy()
    df['Sentence'] = df['Sentence'].apply(clean_text)
    df['# Words'] = df['Sentence'].apply(lambda x: len(str(x).split()))
    df['Sentence'] = df['Sentence'].apply(remove_stopwords)

    text_list = df['Sentence'].tolist()
    tokenized = lemmatization(text_list)

    bigram  = Phrases(tokenized, min_count=3, threshold=10)
    bigram_mod  = Phraser(bigram)
    tokenized = [bigram_mod[doc] for doc in tokenized]

    dictionary = corpora.Dictionary(tokenized)
    doc_term_matrix = [dictionary.doc2bow(sent) for sent in tokenized]

    return df, tokenized, dictionary, doc_term_matrix

preprocessed_wayback = {label: preprocess(df) for label, df in groups_wayback.items()}
preprocessed_org = {label: preprocess(df) for label, df in groups_org.items()}
preprocessed_combined = {label: preprocess(df) for label, df in groups_combined.items()}

LDA

In [20]:
def train_lda(dtm, dictionary, label, num_topics=5):
    if not dtm:
        print(f"[{label}] Empty doc-term matrix, skipping.")
        return None, None
    model = gensim.models.ldamodel.LdaModel(
        corpus=dtm, id2word=dictionary,
        num_topics=num_topics, random_state=100,
        chunksize=1000, passes=50, iterations=100
    )
    vis = gensimvis.prepare(model, dtm, dictionary)
    return model, vis

lda_wayback = {
    label: train_lda(v[3], v[2], label, 5)
    for label, v in preprocessed_wayback.items()
}
lda_org = {
    label: train_lda(v[3], v[2], label, 5)
    for label, v in preprocessed_org.items()
}
# Use fewer topics when combining org and time period (less support)
lda_combined = {
    label: train_lda(v[3], v[2], label, 3)
    for label, v in preprocessed_combined.items()
}

Visualizations

In [23]:
lda_wayback["Wayback"][1]

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
3      0.108213  0.106103       1        1  32.288857
4      0.077215  0.168944       2        1  22.444856
1     -0.001764 -0.008053       3        1  17.682545
0      0.124910 -0.246672       4        1  17.513451
2     -0.308574 -0.020322       5        1  10.070291, topic_info=           Term        Freq       Total Category  logprob  loglift
131  technology  274.000000  274.000000  Default  30.0000  30.0000
32       energy  843.000000  843.000000  Default  29.0000  29.0000
27   production  329.000000  329.000000  Default  28.0000  28.0000
44      natural  319.000000  319.000000  Default  27.0000  27.0000
4          cost  145.000000  145.000000  Default  26.0000  26.0000
..          ...         ...         ...      ...      ...      ...
32       energy   58.210423  843.329955   Topic5  -4.2078  -0.3777
327     service   27.796051   99.370196   Topic5  -4.9470   1.0216
5      customer   23.716898  152.853157   Topic5  -5.1057   0.4323
128    facility   22.189059  218.063605   Topic5  -5.1723   0.0104
205        fuel   19.257106  113.458465   Topic5  -5.3140   0.5220

[310 rows x 6 columns], token_table=      Topic      Freq         Term
term                              
73        3  0.945704  acquisition
306       1  0.988946       action
810       3  0.896771     addition
810       5  0.091977     addition
1         5  0.966390    advantage
...     ...       ...          ...
63        1  0.437669         year
63        2  0.085631         year
63        3  0.304465         year
63        4  0.104660         year
63        5  0.066602         year

[404 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[4, 5, 2, 1, 3])

In [25]:
lda_wayback["Non-Wayback"][1]

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
2     -0.147714  0.098512       1        1  33.977615
3     -0.031394  0.010391       2        1  18.212006
0     -0.069350  0.011638       3        1  16.840094
4      0.029580 -0.222236       4        1  16.576310
1      0.218878  0.101694       5        1  14.393976, topic_info=                Term        Freq       Total Category  logprob  loglift
35            energy  381.000000  381.000000  Default  30.0000  30.0000
43         renewable  109.000000  109.000000  Default  29.0000  29.0000
24            carbon   61.000000   61.000000  Default  28.0000  28.0000
151           global   86.000000   86.000000  Default  27.0000  27.0000
33   chief_executive   39.000000   39.000000  Default  26.0000  26.0000
..               ...         ...         ...      ...      ...      ...
64           project   19.667852  288.319699   Topic5  -4.7947  -0.7467
182          natural   15.217557  147.765247   Topic5  -5.0513  -0.3348
72              year   13.841111   78.506494   Topic5  -5.1461   0.2028
358           system   12.676967   46.755014   Topic5  -5.2339   0.6332
4             growth   12.819585   73.148740   Topic5  -5.2228   0.1968

[333 rows x 6 columns], token_table=      Topic      Freq      Term
term                           
580       3  0.983495      able
534       2  0.952971    action
449       1  0.315764  activity
449       2  0.552587  activity
449       3  0.078941  activity
...     ...       ...       ...
72        1  0.076427      year
72        2  0.318445      year
72        3  0.229280      year
72        4  0.216543      year
72        5  0.178329      year

[486 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[3, 4, 1, 5, 2])

In [21]:
lda_combined["Suncor Energy | Wayback"][1]

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0      0.126000  0.099141       1        1  38.407202
1      0.025404 -0.155548       2        1  31.873714
2     -0.151404  0.056407       3        1  29.719084, topic_info=            Term        Freq       Total Category  logprob  loglift
118   technology  118.000000  118.000000  Default  30.0000  30.0000
25    production  147.000000  147.000000  Default  29.0000  29.0000
144  development   51.000000   51.000000  Default  28.0000  28.0000
341       member   30.000000   30.000000  Default  27.0000  27.0000
384        focus   32.000000   32.000000  Default  26.0000  26.0000
..           ...         ...         ...      ...      ...      ...
63      business   22.425919   98.153364   Topic3  -5.0881  -0.2629
66    investment   19.690737   68.188834   Topic3  -5.2182  -0.0288
264   commitment   20.410681   97.524685   Topic3  -5.1823  -0.3507
47       project   21.622712  139.863277   Topic3  -5.1246  -0.6535
19          work   15.716508   32.095202   Topic3  -5.4436   0.4994

[207 rows x 6 columns], token_table=      Topic      Freq         Term
term                              
320       2  0.966703      ability
703       1  0.931849      advance
253       1  0.904207     albertas
91        3  0.913872  alternative
663       1  0.939659       annual
...     ...       ...          ...
86        2  0.585941        world
86        3  0.103401        world
22        1  0.387314         year
22        2  0.202077         year
22        3  0.420993         year

[241 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[1, 2, 3])

In [22]:
lda_combined["Suncor Energy | Non-Wayback"][1]

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0      0.107790 -0.057781       1        1  45.688312
2     -0.115164 -0.047349       2        1  29.956441
1      0.007375  0.105129       3        1  24.355246, topic_info=                   Term      Freq     Total Category  logprob  loglift
69      power_renewable  6.000000  6.000000  Default  30.0000  30.0000
68   petroleum_resource  6.000000  6.000000  Default  29.0000  29.0000
67    future_investment  6.000000  6.000000  Default  28.0000  28.0000
51                 fuel  7.000000  7.000000  Default  27.0000  27.0000
92          performance  8.000000  8.000000  Default  26.0000  26.0000
..                  ...       ...       ...      ...      ...      ...
104            facility  0.845417  2.005614   Topic3  -5.3165   0.5485
46              company  0.858512  8.955690   Topic3  -5.3012  -0.9324
138       environmental  0.848837  6.359885   Topic3  -5.3125  -0.6015
184             capital  0.833272  2.656033   Topic3  -5.3310   0.2532
30                early  0.832922  2.655507   Topic3  -5.3314   0.2530

[167 rows x 6 columns], token_table=      Topic      Freq       Term
term                            
114       1  0.716031     action
114       3  0.238677     action
41        2  0.762084    advance
284       3  0.779287     affair
42        2  0.392160   alliance
...     ...       ...        ...
99        1  0.840699   upstream
113       1  0.723099      value
113       3  0.180775      value
292       3  0.779287  westcoast
148       2  1.020350      world

[180 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[1, 3, 2])

In [26]:
lda_combined["Canadian Natural Resources | Wayback"][1]

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
1     -0.184546  0.026113       1        1  42.170130
2      0.062364 -0.133896       2        1  30.783048
0      0.122182  0.107783       3        1  27.046822, topic_info=               Term       Freq      Total Category  logprob  loglift
19         emission  96.000000  96.000000  Default  30.0000  30.0000
25        reduction  72.000000  72.000000  Default  29.0000  29.0000
129            year  33.000000  33.000000  Default  28.0000  28.0000
79       production  60.000000  60.000000  Default  27.0000  27.0000
85   carbon_capture  25.000000  25.000000  Default  26.0000  26.0000
..              ...        ...        ...      ...      ...      ...
22       government  10.763169  40.602761   Topic3  -4.7880  -0.0201
40        important   8.044445  16.011017   Topic3  -5.0792   0.6193
62          natural  12.117283  70.021130   Topic3  -4.6695  -0.4466
12        operation   9.102275  34.280597   Topic3  -4.9557  -0.0185
115            well   8.262541  21.647267   Topic3  -5.0524   0.3445

[186 rows x 6 columns], token_table=      Topic      Freq         Term
term                              
352       2  0.933391  abandonment
316       3  0.800045      ability
37        1  0.911961     absolute
66        2  0.933458     activity
138       2  0.160133   additional
...     ...       ...          ...
133       2  0.856976        water
115       2  0.600538         well
115       3  0.369562         well
129       2  0.810751         year
129       3  0.180167         year

[216 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[2, 3, 1])

In [27]:
lda_combined["Canadian Natural Resources | Non-Wayback"][1]

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
1      0.128036 -0.037089       1        1  38.312859
0     -0.102179 -0.074786       2        1  34.515110
2     -0.025857  0.111875       3        1  27.172032, topic_info=                Term      Freq      Total Category  logprob  loglift
10             asset  9.000000   9.000000  Default  30.0000  30.0000
53           capital  9.000000   9.000000  Default  29.0000  29.0000
58            strong  9.000000   9.000000  Default  28.0000  28.0000
33              rate  4.000000   4.000000  Default  27.0000  27.0000
22      high_quality  4.000000   4.000000  Default  26.0000  26.0000
..               ...       ...        ...      ...      ...      ...
31             price  2.263274   5.652972   Topic3  -4.7949   0.3876
20  canadian_natural  2.926584  12.878452   Topic3  -4.5379  -0.1787
18        technology  2.294080   6.401163   Topic3  -4.7814   0.2768
71        production  3.023822  18.626806   Topic3  -4.5052  -0.5151
58            strong  2.463097   9.217986   Topic3  -4.7103  -0.0168

[171 rows x 6 columns], token_table=      Topic      Freq         Term
term                              
141       3  0.969160  acquisition
19        3  0.729943         acre
269       1  0.685484       action
104       2  0.928682     activity
9         2  0.287138    advantage
...     ...       ...          ...
79        1  0.348279         well
79        2  0.348279         well
79        3  0.174139         well
42        1  0.599162         work
42        3  0.399441         work

[184 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[2, 1, 3])

In [28]:
lda_combined["Imperial Oil | Wayback"][1]

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
2     -0.150991 -0.002019       1        1  52.612756
0      0.077544 -0.110590       2        1  23.929270
1      0.073446  0.112609       3        1  23.457974, topic_info=                    Term       Freq      Total Category  logprob  loglift
17            technology  64.000000  64.000000  Default  30.0000  30.0000
2    greenhouse_emission  61.000000  61.000000  Default  29.0000  29.0000
122           production  42.000000  42.000000  Default  28.0000  28.0000
256             canadian  14.000000  14.000000  Default  27.0000  27.0000
487              program  17.000000  17.000000  Default  26.0000  26.0000
..                   ...        ...        ...      ...      ...      ...
26                future   5.804163  24.056901   Topic3  -5.1806   0.0281
187          development   5.394670  19.843151   Topic3  -5.2538   0.1475
124              project   6.400365  53.260876   Topic3  -5.0828  -0.6689
7       renewable_diesel   5.809323  39.401976   Topic3  -5.1797  -0.4644
96              emission   5.046707  59.742222   Topic3  -5.3204  -1.0213

[202 rows x 6 columns], token_table=      Topic      Freq       Term
term                            
367       1  0.161238  agreement
367       3  0.806188  agreement
142       1  0.150586   alliance
142       2  0.301173   alliance
142       3  0.602346   alliance
...     ...       ...        ...
251       3  0.774485    vehicle
411       2  0.352728       work
411       3  0.587879       work
203       1  0.609446       year
203       2  0.375044       year

[250 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[3, 1, 2])

In [29]:
lda_combined["Imperial Oil | Non-Wayback"][1]

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0      0.084622  0.086059       1        1  38.469012
1     -0.134044  0.016512       2        1  30.843807
2      0.049421 -0.102571       3        1  30.687182, topic_info=                Term      Freq     Total Category  logprob  loglift
48           project  6.000000  6.000000  Default  30.0000  30.0000
72           product  6.000000  6.000000  Default  29.0000  29.0000
2    large_renewable  7.000000  7.000000  Default  28.0000  28.0000
1    diesel_facility  7.000000  7.000000  Default  27.0000  27.0000
42              year  5.000000  5.000000  Default  26.0000  26.0000
..               ...       ...       ...      ...      ...      ...
130           volume  2.229024  8.281902   Topic3  -4.5078  -0.1312
27        leadership  1.576870  2.741552   Topic3  -4.8540   0.6282
57        efficiency  1.574420  2.741693   Topic3  -4.8555   0.6266
8             centre  1.576627  4.165700   Topic3  -4.8541   0.2097
25        greenhouse  1.571099  2.741903   Topic3  -4.8576   0.6244

[169 rows x 6 columns], token_table=      Topic      Freq        Term
term                             
18        3  0.736457      active
52        3  0.985200    activity
53        2  0.369984  additional
53        3  0.739968  additional
64        2  0.984609  advantaged
...     ...       ...         ...
130       1  0.482981      volume
130       2  0.241490      volume
130       3  0.241490      volume
42        2  0.185475        year
42        3  0.741902        year

[173 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[1, 2, 3])

In [30]:
lda_combined["Shell Canada | Wayback"][1]

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0      0.092329  0.085279       1        1  42.554936
1     -0.133620  0.024884       2        1  28.958971
2      0.041291 -0.110163       3        1  28.486093, topic_info=              Term       Freq      Total Category  logprob  loglift
9          vehicle  43.000000  43.000000  Default  30.0000  30.0000
181        company  27.000000  27.000000  Default  29.0000  29.0000
142           team  19.000000  19.000000  Default  28.0000  28.0000
41            sand  28.000000  28.000000  Default  27.0000  27.0000
22   hydrogen_fuel  17.000000  17.000000  Default  26.0000  26.0000
..             ...        ...        ...      ...      ...      ...
224     innovation   6.781857  18.151139   Topic3  -5.2932   0.2713
182        country   6.552001  16.296462   Topic3  -5.3276   0.3446
154        natural   7.251010  24.437222   Topic3  -5.2263   0.0408
149     government   7.142218  23.169918   Topic3  -5.2414   0.0789
3         emission   6.886475  50.870787   Topic3  -5.2778  -0.7440

[200 rows x 6 columns], token_table=      Topic      Freq         Term
term                              
106       1  0.826397  achievement
261       3  0.965414       agency
491       2  0.802982      alberta
491       3  0.160596      alberta
482       2  0.770347       amount
...     ...       ...          ...
42        3  0.298518        world
271       3  0.773009    worldwide
52        1  0.860804         year
52        2  0.137729         year
1146      2  0.917777  year_people

[230 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[1, 2, 3])

In [32]:
lda_combined["Shell Canada | Non-Wayback"][1]

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
1      0.138977 -0.030389       1        1  54.698630
0     -0.106387 -0.070650       2        1  25.139688
2     -0.032590  0.101039       3        1  20.161683, topic_info=           Term       Freq      Total Category  logprob  loglift
12       target   3.000000   3.000000  Default  30.0000  30.0000
0      emission  19.000000  19.000000  Default  29.0000  29.0000
116       group   3.000000   3.000000  Default  28.0000  28.0000
17       carbon  18.000000  18.000000  Default  27.0000  27.0000
2    production   8.000000   8.000000  Default  26.0000  26.0000
..          ...        ...        ...      ...      ...      ...
0      emission   2.335420  19.144915   Topic3  -3.2608  -0.5025
26       market   0.799748   2.661328   Topic3  -4.3325   0.3991
15     upgrader   0.798957   2.664351   Topic3  -4.3335   0.3970
7       outlook   0.792524   2.786249   Topic3  -4.3416   0.3442
58     chemical   0.789812   4.084955   Topic3  -4.3450  -0.0419

[148 rows x 6 columns], token_table=      Topic      Freq          Term
term                               
126       1  0.699069    additional
94        2  0.763400     agreement
56        1  0.494577  announcement
56        3  0.494577  announcement
70        3  0.794199         asian
...     ...       ...           ...
15        1  0.375326      upgrader
15        2  0.375326      upgrader
15        3  0.375326      upgrader
101       2  0.763400      vigeveno
112       3  0.794158          well

[131 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[2, 1, 3])